# Part 2: Feature Engineering, Model Training & Future Forecasting
## Sales & Demand Forecasting for Businesses
**Internship Task 1 (Future Interns)**  

### Objective:
In this notebook, we transform daily sales time series into a rich feature matrix with calendar flags, autoregressive lags, and rolling statistical indicators. We train multiple forecasting models (Baseline, Ridge, Random Forest, XGBoost), evaluate them against out-of-time test data, and project the next 30 days of customer demand.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_engineering import prepare_modeling_dataset
from src.model import get_models, train_and_evaluate_all, generate_future_forecast

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

### 1. Load Daily Sales & Construct Temporal Features
**Key Architecture Note:** Lags and rolling aggregates are computed strictly on shifted past values ($t-1$) to guarantee **zero future data leakage**.

In [ ]:
daily_path = os.path.join("..", "data", "processed", "daily_sales.csv")
daily_df = pd.read_csv(daily_path)
daily_df['order_date'] = pd.to_datetime(daily_df['order_date'])

clean_df, feature_cols, train_df, test_df, X_train, y_train, X_test, y_test = prepare_modeling_dataset(
    daily_df, target_col='total_sales', test_days=90
)
print(f"Feature count: {len(feature_cols)}")
print(f"Training instances (days): {len(train_df)}")
print(f"Testing instances (days):  {len(test_df)}")
print("Sample features:", feature_cols[:10])

### 2. Train Models & Evaluate on 90-Day Out-of-Time Holdout Set

In [ ]:
models = get_models(random_state=42)
summary_df, fitted_models, predictions = train_and_evaluate_all(models, X_train, y_train, X_test, y_test)
summary_df

### 3. Test Set Forecast Comparison

In [ ]:
best_model_name = summary_df.iloc[0]['Model']
print(f"Winning Model: {best_model_name}")

plt.figure(figsize=(14, 5))
plt.plot(test_df['order_date'], test_df['total_sales'], label='Actual Sales', color='#37474F', linewidth=1.8)
plt.plot(test_df['order_date'], predictions[best_model_name], label=f'Forecast ({best_model_name})', color='#E65100', linewidth=2.2, linestyle='--')
plt.title(f'90-Day Holdout Evaluation: Actual vs. Forecast ({best_model_name})', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()

### 4. Future 30-Day Forward Projection with Safety Stock Bounds

In [ ]:
best_model = fitted_models[best_model_name]
forecast_df = generate_future_forecast(best_model, daily_df, forecast_days=30, service_level_z=1.65)
forecast_df.head(10)

### 5. Final Future Projection Plot for Retail Stakeholders

In [ ]:
recent_tail = daily_df.tail(60)
plt.figure(figsize=(14, 5))
plt.plot(recent_tail['order_date'], recent_tail['total_sales'], label='Recent Actual Sales', color='#37474F')
plt.plot(forecast_df['date'], forecast_df['predicted_sales'], label='Projected Demand (Next 30 Days)', color='#00897B', linewidth=2.5, marker='s', markersize=4)
plt.fill_between(forecast_df['date'], forecast_df['conservative_lower'], forecast_df['safety_stock_upper'], color='#80CBC4', alpha=0.35, label='Safety Stock Buffer (90% Service Level)')
plt.title('Future 30-Day Demand Projection with Recommended Safety Stock Thresholds', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Projected Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()